# Index options: risk as of the reporting date vs re-risked on live index spreads

Three tables for all live CDX / iTraxx options in the **Credit** and **Manual** strategies (Abhishek's trades):
1. **As of `REPORT_DATE`**: index spread, forward spread, implied vol, PV, delta, gamma, vega, theta, from the pricer cache (the spreads Quantifi used for that date's risk).
2. **LIVE**: same columns, regenerated on live Bloomberg index spreads (`bbg_utils`: `CDX IG CDSI S<n> 5Y Corp`, `ITRX EUR CDSI S<n> 5Y Corp`). **Implied vols are held constant** (sticky strike); the valuation date stays `REPORT_DATE`, so theta is the 1-day theta at the new spread and no time decay is applied.
3. **Changes** (live - report date).

Conventions (option's own currency, not FX converted): `delta` = Toolkit MarketDelta (ratio), `delta_notional` = delta x Notional (index-notional hedge); `gamma_pv` = PV(s+1bp) - 2 PV(s) + PV(s-1bp) on index-quote bumps (currency per bp^2, comparable to the official 'Credit Spread Gamma (1bp)'); `gamma_dn` = change in `delta_notional` per +1bp (the Toolkit's MarketGamma ratio is not per-bp, so it is not shown); `vega` = PV change for +1 vol pt; `theta` = 1-day MarketTheta; vega and theta already include notional.
`Credit` (official booking) and `Manual` hold the same trades twice: do not add the two strategies together.

Needs the pricer cache for `REPORT_DATE`: `python pricer_cache.py build --as-of 2026-11-16 --live-only --strategies Credit,Manual`. **Restart the kernel after any qncore install.**

In [ ]:
import os
import pandas as pd
import requests
from Quantifi.Toolkit.Base import Dt            # Toolkit import must come before qservices
from qservices.QuantifiApiClient import Connect
import pricer_cache as pc
import option_repricer as opr

ENV = 'Prod'
host = {'Prod': 'hb-dc1pd-qfidsp', 'UAT': 'hb-dc1ut-qfidsp'}[ENV]
os.environ['Q_DATA_SERVICE_URL_PREFIX'] = f'https://{host}:47200'
os.environ['Q_MARKET_ENV_SERVICE_URL_PREFIX'] = f'https://{host}:47201'
os.environ['Q_LOCAL_INSTALL'] = 'True'
requests.packages.urllib3.disable_warnings()

REPORT_DATE = '2026-11-16'
CALC_ENV = 'Official'
STRATEGIES = ['Credit', 'Manual']
TRADERS = ['abhutra', 'AB', 'Abhishek Bhutra']
# 'level'  : reprice at the Bloomberg live spread as is
# 'change' : reprice at Quantifi's booked spread + (Bloomberg live - Bloomberg close on REPORT_DATE)  (removes source basis)
SPREAD_MODE = 'level'

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
pd.options.display.float_format = '{:,.2f}'.format

STATIC = ['TradeId', 'strategy', 'index_name', 'type', 'strike_bp', 'expiry', 'notional']
RISK = ['index_spread_bp', 'fwd_spread_bp', 'implied_vol', 'pv', 'delta', 'delta_notional',
        'gamma_pv', 'gamma_dn', 'vega', 'theta']

## Load options and pricers

In [ ]:
tl = ', '.join(f"'{t}'" for t in TRADERS)
all_trades = Connect.data_query(f"FROM Trade b WHERE b.Trader.Name IN ({tl})", max_results=100_000, verify_https_cert=False)
opts = opr.option_table(all_trades, REPORT_DATE, STRATEGIES)
pricers = opr.load_option_pricers(opts, REPORT_DATE, CALC_ENV)
print(f'{len(opts)} live options at {REPORT_DATE} in {STRATEGIES}; {len(pricers)} pricers loaded')

## 1. Risk as of the reporting date (cached pricers: spreads and vols as of `REPORT_DATE`)

In [ ]:
base = opts.merge(opr.risk_table(opts, pricers), on='TradeId')
table_report = base[STATIC + RISK]
table_report

## Live index spreads (Bloomberg via `bbg_utils`)

In [ ]:
live = opr.live_spreads(opts.index_name.unique())
booked_bp = {r.index_name: float(pricers[r.TradeId].Spread) * 1e4 for r in opts.itertuples()}
live['report_date_bp'] = live.index_name.map(booked_bp)
live['change_bp'] = live.spread_bp - live.report_date_bp

if SPREAD_MODE == 'change':
    new_bp = opr.shifted_spreads(opts, pricers, live, REPORT_DATE)
else:
    new_bp = dict(zip(live.index_name, live.spread_bp))
live['spread_used_bp'] = live.index_name.map(new_bp)
print('Spread mode:', SPREAD_MODE)
live

## 2. LIVE risk (re-generated on the live index spreads, same implied vols)

In [ ]:
live_df = opts.merge(opr.risk_table(opts, pricers, spreads_bp=new_bp), on='TradeId')
table_live = live_df[STATIC + RISK]
table_live

## 3. Changes (live - report date)

In [ ]:
b = table_report.set_index('TradeId'); l = table_live.set_index('TradeId')
table_chg = opts.set_index('TradeId')[['strategy', 'index_name', 'type', 'strike_bp', 'expiry', 'notional']].join(
    (l[RISK] - b[RISK]).add_prefix('chg_')).reset_index()
table_chg

### Summary by index (Manual strategy; Credit holds the same trades)

In [ ]:
m = table_report[table_report.strategy == 'Manual'].set_index('TradeId')
lv = table_live[table_live.strategy == 'Manual'].set_index('TradeId')
cols = ['pv', 'delta_notional', 'gamma_pv', 'gamma_dn', 'vega', 'theta']
summ = pd.concat({'report_date': m.groupby('index_name')[cols].sum(), 'live': lv.groupby('index_name')[cols].sum()}, axis=1)
summ

In [ ]:
out = f'options_risk_{REPORT_DATE}_live.xlsx'
with pd.ExcelWriter(out) as xw:
    table_report.to_excel(xw, sheet_name='report_date', index=False)
    table_live.to_excel(xw, sheet_name='live', index=False)
    table_chg.to_excel(xw, sheet_name='changes', index=False)
    live.to_excel(xw, sheet_name='live_spreads', index=False)
print('saved', out)